# ARMA Model Tutorial

This notebook demonstrates how to fit an **AutoRegressive Moving Average (ARMA)** model
using the `statsmodels` library, while leveraging the helper functions in this
repository's `timeseries` module for data exploration.

## What is ARMA?

An ARMA(p, q) model combines:
- **AR(p)** — the value depends on *p* past values (autoregressive part)
- **MA(q)** — the value depends on *q* past forecast errors (moving-average part)

$$X_t = c + \sum_{i=1}^{p} \phi_i X_{t-i} + \sum_{j=1}^{q} \theta_j \varepsilon_{t-j} + \varepsilon_t$$

ARMA assumes the series is **stationary**. If it is not, we first difference the
series (giving an ARIMA model).

## Dataset

We use the **Airline Passengers** dataset (Box & Jenkins, 1976) — a classic
open-source time-series dataset shipped with `statsmodels`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from timeseries import moving_average, difference, autocorrelation

## 1. Load and visualise the data

In [ ]:
import statsmodels.api as sm

data = sm.datasets.co2.load_pandas().data  # Mauna Loa CO2 — monthly, stationary after differencing
data = data.resample("ME").mean().ffill()   # fill any gaps
series = data["co2"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series)
ax.set_title("Mauna Loa Weekly CO2 Concentrations (monthly average)")
ax.set_ylabel("CO2 (ppm)")
plt.tight_layout()
plt.show()

## 2. Explore with library helpers

Use the repository's own `moving_average`, `difference`, and `autocorrelation`
functions to understand the series before modelling.

In [ ]:
values = series.values.tolist()

# Moving average (window = 12 months)
ma12 = moving_average(values, 12)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.index, values, alpha=0.5, label="Original")
ax.plot(series.index, ma12, label="12-month MA", linewidth=2)
ax.set_title("CO2 with 12-Month Moving Average")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Differencing to make the series stationary
diff1 = difference(values, lag=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(diff1)
ax.set_title("First-Order Differenced CO2 Series")
ax.axhline(0, color="grey", linestyle="--")
plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation at various lags using our library function
lags = range(1, 25)
acf_values = [autocorrelation(diff1, lag) for lag in lags]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lags, acf_values)
ax.set_xlabel("Lag")
ax.set_ylabel("Autocorrelation")
ax.set_title("ACF of Differenced CO2 Series (computed with timeseries.autocorrelation)")
plt.tight_layout()
plt.show()

## 3. Fit an ARMA model

We fit the model on the **differenced** series (equivalent to ARIMA(p,1,q) on
the original). Based on the ACF/PACF we start with ARMA(2,2).

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# ARIMA(p,d,q) with d=1 is equivalent to ARMA(p,q) on the differenced series
model = ARIMA(series, order=(2, 1, 2))
results = model.fit()
print(results.summary())

## 4. Diagnostics

In [ ]:
results.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

## 5. Forecasting

In [ ]:
forecast = results.get_forecast(steps=24)
fc_mean = forecast.predicted_mean
fc_ci = forecast.conf_int()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(series[-120:], label="Observed")
ax.plot(fc_mean, label="Forecast", color="red")
ax.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                color="red", alpha=0.15, label="95% CI")
ax.set_title("ARMA(2,2) — 24-Month Forecast")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Step | Tool / Function |
|---|---|
| Trend inspection | `timeseries.moving_average` |
| Stationarity via differencing | `timeseries.difference` |
| ACF analysis | `timeseries.autocorrelation` |
| ARMA fitting & forecasting | `statsmodels.tsa.arima.model.ARIMA` |